In [ ]:
import pandas as pd, os

def p(n):
    return next(x for x in ["/" + n, "/content/" + n] if os.path.exists(x))

def clean(d, cols):
    d = d.iloc[:, :4]
    d.columns = cols
    d["HS"] = pd.to_numeric(d["HS"], errors="coerce")
    d = d.dropna(subset=["HS"])
    d["HS"] = d["HS"].astype(int).astype(str).str.zfill(2)
    for c in cols[2:]:
        d[c] = pd.to_numeric(d[c].astype(str).str.replace(",", ""), errors="coerce")
    return d

files = {"China": "imp_china..xlsx", "Russia": "imp_Russia.xlsx", "USA": "imp_US.xlsx"}

df = clean(pd.read_excel(p("world_imports25_26.xlsx")),
           ["HS", "Commodity", "W_2425", "W_2526"])

for c, f in files.items():
    d = clean(pd.read_excel(p(f)),
              ["HS", "Commodity", f"{c}_2425", f"{c}_2526"])
    df = df.merge(d.drop(columns="Commodity"), on="HS", how="left")

df = df.fillna(0)

for y in ["2425", "2526"]:
    for c in files:
        df[f"{c}_sh{y}"] = df[f"{c}_{y}"] / df[f"W_{y}"]
    sh = [f"{c}_sh{y}" for c in files]
    df[f"top_sh{y}"] = df[sh].max(axis=1)
    df[f"top_c{y}"] = df[sh].idxmax(axis=1).str.split("_").str[0]

print(df.shape)
dep = df[df["top_sh2526"] > 0.5].sort_values("W_2526", ascending=False)
print(dep[["HS", "Commodity", "W_2526", "top_c2526", "top_sh2526"]].head(10))
print("Share of imports in single-supplier products:",
      round(dep["W_2526"].sum() / df["W_2526"].sum() * 100, 1), "%")

(98, 20)
    HS                                          Commodity   W_2526 top_c2526  \
69  70                               GLASS AND GLASSWARE.  3123.39     China   
92  94  FURNITURE; BEDDING, MATTRESSES, MATTRESS SUPPO...  1964.97     China   
53  54                                MAN-MADE FILAMENTS.  1958.20     China   
67  68  ARTICLES OF STONE, PLASTER, CEMENT, ASBESTOS, ...  1162.76     China   
96  98                  PROJECT GOODS; SOME SPECIAL USES.  1112.84    Russia   
68  69                                  CERAMIC PRODUCTS.  1043.60     China   
59  60                      KNITTED OR CROCHETED FABRICS.   950.09     China   
58  59  IMPREGNATED, COATED, COVERED OR LAMINATED TEXT...   937.98     China   
84  86  RAILWAY OR TRAMWAY LOCOMOTIVES, ROLLING-STOCK ...   808.48     China   
94  96               MISCELLANEOUS MANUFACTURED ARTICLES.   646.11     China   

    top_sh2526  
69    0.579691  
92    0.529927  
53    0.729262  
67    0.504102  
96    0.568752  
68    0.

In [ ]:
n = df[df["top_sh2526"] > 0.5]
print(n.groupby("top_c2526")["W_2526"].agg(["count", "sum"]).round(0))
print("Share of India's imports in single-supplier products:",
      round(n["W_2526"].sum() / df["W_2526"].sum() * 100, 1), "%")

df.to_csv("dependency.csv", index=False)

           count      sum
top_c2526                
China         16  14266.0
Russia         1   1113.0
Share of India's imports in single-supplier products: 2.0 %


In [ ]:
tot = df["W_2526"].sum()

for t in [0.3, 0.4, 0.5]:
    m = df[df["top_sh2526"] > t]
    print(f"Above {int(t*100)}%:", len(m), "products,",
          round(m["W_2526"].sum() / tot * 100, 1), "% of imports")

for c in ["China", "Russia", "USA"]:
    print(c, round(df[f"{c}_2526"].sum() / tot * 100, 1), "% of India's imports")

top10 = df.sort_values("W_2526", ascending=False).head(10)
print(top10[["HS", "Commodity", "W_2526", "top_c2526", "top_sh2526"]])

Above 30%: 38 products, 34.6 % of imports
Above 40%: 24 products, 20.0 % of imports
Above 50%: 17 products, 2.0 % of imports
China 17.0 % of India's imports
Russia 7.1 % of India's imports
USA 6.9 % of India's imports
    HS                                          Commodity     W_2526  \
26  27  MINERAL FUELS, MINERAL OILS AND PRODUCTS OF TH...  203415.63   
70  71  NATURAL OR CULTURED PEARLS,PRECIOUS OR SEMIPRE...  109429.22   
83  85  ELECTRICAL MACHINERY AND EQUIPMENT AND PARTS T...  104824.72   
82  84  NUCLEAR REACTORS, BOILERS, MACHINERY AND MECHA...   74002.16   
28  29                                  ORGANIC CHEMICALS   25410.73   
38  39                      PLASTIC AND ARTICLES THEREOF.   22232.50   
14  15  ANIMAL OR VEGETABLE FATS AND OILS AND THEIR CL...   19776.97   
71  72                                     IRON AND STEEL   15752.90   
88  90  OPTICAL, PHOTOGRAPHIC CINEMATOGRAPHIC MEASURIN...   15352.87   
30  31                                       FERTILISERS.   14